In [ ]:
"""
Factor Analysis with K-Fold Cross Validation
"""

# Google Colab setup
try:
    from google.colab import drive
    drive.mount('/content/drive')
    # Modify this path according to your Google Drive structure
    folder_path = "/content/drive/My Drive/Factordata"  # Update this path!
    print("Google Drive mounted successfully")
except:
    print("Not running in Google Colab or drive mount failed")
    folder_path = "."  # Default to current directory if not in Colab

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, cross_val_score, KFold, cross_validate
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler

# Install required packages if in Colab
try:
    import google.colab
    !pip install imbalanced-learn
    !pip install openpyxl
except:
    pass

def process_one_measure(df, measure):
    """
    Add rolling statistics for both window sizes.
    """
    for window in [12, 24]:
        df[f'{measure}_Rolling_Mean_{window}'] = df[measure].rolling(window=window, min_periods=1).mean()
        df[f'{measure}_Rolling_Std_{window}'] = df[measure].rolling(window=window, min_periods=1).std()
        df[f'{measure}_Dynamic_Outlier_{window}'] = (
            (df[measure] - df[f'{measure}_Rolling_Mean_{window}']).abs() > 2 * df[f'{measure}_Rolling_Std_{window}']
        )
    return df

def plot_factor_analysis(df, measure, save_path):
    """
    Create and save analysis plots for the given measure.
    """
    plt.figure(figsize=(15, 10))

    plt.subplot(2, 2, 1)
    for source in df['Source_File'].unique():
        fund_data = df[df['Source_File'] == source]
        plt.plot(fund_data.index, fund_data[measure], label=source, alpha=0.4)
        outliers = fund_data[fund_data[f'{measure}_Dynamic_Outlier_12']]
        plt.scatter(outliers.index, outliers[measure], marker='x', alpha=0.6)
    plt.title(f'{measure} Values Over Time (Window=12)')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.xticks(rotation=45)

    plt.subplot(2, 2, 2)
    for source in df['Source_File'].unique():
        fund_data = df[df['Source_File'] == source]
        plt.plot(fund_data.index, fund_data[f'{measure}_Rolling_Mean_12'], alpha=0.4)
        plt.plot(fund_data.index, fund_data[f'{measure}_Rolling_Mean_24'], alpha=0.4, linestyle='dashed')
    plt.title(f'{measure} Rolling Mean by Fund (12 & 24)')
    plt.xticks(rotation=45)

    plt.subplot(2, 2, 3)
    sns.histplot(data=df, x=measure, hue='Source_File', multiple="layer", alpha=0.4)
    plt.title(f'{measure} Distribution by Fund')

    plt.subplot(2, 2, 4)
    for source in df['Source_File'].unique():
        fund_data = df[df['Source_File'] == source]
        monthly_outliers_12 = fund_data[f'{measure}_Dynamic_Outlier_12'].resample('ME').mean()
        monthly_outliers_24 = fund_data[f'{measure}_Dynamic_Outlier_24'].resample('ME').mean()
        plt.plot(monthly_outliers_12.index, monthly_outliers_12, label=f'{source} (Window=12)', alpha=0.6)
        plt.plot(monthly_outliers_24.index, monthly_outliers_24, label=f'{source} (Window=24)', alpha=0.6, linestyle='dashed')
    plt.title('Monthly Outlier Frequency by Fund (12 & 24)')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.xticks(rotation=45)

    plt.tight_layout()
    plt.savefig(os.path.join(save_path, f'{measure}_analysis.png'), bbox_inches='tight')
    plt.close()

def process_file(file_path, measure):
    """
    Process a single Excel file and extract relevant data.
    """
    try:
        df = pd.read_excel(file_path, sheet_name="FactorOutput")
        if 'Date' in df.columns:
            df['Date'] = pd.to_datetime(df['Date'])
        else:
            raise ValueError("The 'Date' column is missing in the file.")

        df = process_one_measure(df, measure)
        df['Source_File'] = os.path.basename(file_path).replace('.xlsx', '')
        return df
    except Exception as e:
        print(f"Error processing file {file_path}: {e}")
        return None

def perform_kfold_analysis(X, y, clf, n_splits=5):
    """
    Perform k-fold cross validation with multiple metrics.
    """
    scoring = {
        'accuracy': 'accuracy',
        'precision': 'precision',
        'recall': 'recall',
        'f1': 'f1',
        'roc_auc': 'roc_auc'
    }

    kfold = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    cv_results = cross_validate(clf, X, y, cv=kfold, scoring=scoring, return_train_score=True)

    results = {
        'test_accuracy': cv_results['test_accuracy'].mean(),
        'test_accuracy_std': cv_results['test_accuracy'].std(),
        'test_precision': cv_results['test_precision'].mean(),
        'test_precision_std': cv_results['test_precision'].std(),
        'test_recall': cv_results['test_recall'].mean(),
        'test_recall_std': cv_results['test_recall'].std(),
        'test_f1': cv_results['test_f1'].mean(),
        'test_f1_std': cv_results['test_f1'].std(),
        'test_roc_auc': cv_results['test_roc_auc'].mean(),
        'test_roc_auc_std': cv_results['test_roc_auc'].std(),
        'train_accuracy': cv_results['train_accuracy'].mean(),
        'train_precision': cv_results['train_precision'].mean(),
        'train_recall': cv_results['train_recall'].mean(),
        'train_f1': cv_results['train_f1'].mean(),
        'train_roc_auc': cv_results['train_roc_auc'].mean()
    }

    return results

def plot_cv_results(results, measure, name, save_path):
    """
    Create visualization of cross-validation results.
    """
    plt.figure(figsize=(12, 6))
    metrics = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
    train_values = [results[f'train_{m}'] for m in metrics]
    test_values = [results[f'test_{m}'] for m in metrics]
    test_stds = [results[f'test_{m}_std'] for m in metrics]

    x = np.arange(len(metrics))
    width = 0.35

    plt.bar(x - width/2, train_values, width, label='Train', alpha=0.8)
    plt.bar(x + width/2, test_values, width, label='Test', alpha=0.8)
    plt.errorbar(x + width/2, test_values, yerr=test_stds, fmt='none', color='black', capsize=5)

    plt.xlabel('Metrics')
    plt.ylabel('Score')
    plt.title(f'Cross-validation Results - {name} ({measure})')
    plt.xticks(x, metrics, rotation=45)
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(save_path, f'{measure}_{name}_cv_results.png'))
    plt.close()


        # First, let's check what files are in the folder
print("Checking folder contents...")
for filename in os.listdir(folder_path):
    print(f"Found file: {filename}")

def main():
    factors = ['Alpha..annualisiert.', 'Value.Growth', 'Small.Large', 'Momentum', 'Volatility']
    plots_dir = os.path.join(folder_path, 'analysis_plots')
    os.makedirs(plots_dir, exist_ok=True)
    all_results = []

    for measure in factors:
        print(f"\nProcessing measure: {measure}")
        processed_data = []

        print(f"Looking for Excel files in: {folder_path}")
        excel_files = [f for f in os.listdir(folder_path) if f.endswith(".xlsx") and not f.startswith("Model_Comparison")]

        for filename in excel_files:
            file_path = os.path.join(folder_path, filename)
            print(f"\nTrying to process file: {file_path}")
            try:
                df = pd.read_excel(file_path)  # Remove sheet_name parameter
                print(f"Successfully read Excel file. Shape: {df.shape}")
                print(f"Columns found: {df.columns.tolist()}")

                if measure in df.columns:
                    processed_df = process_one_measure(df.copy(), measure)
                    processed_df['Source_File'] = os.path.basename(file_path).replace('.xlsx', '')
                    processed_data.append(processed_df)
                    print(f"Successfully processed file {filename}")
                else:
                    print(f"Measure {measure} not found in {filename}")
            except Exception as e:
                print(f"Error reading file {filename}: {str(e)}")

        if processed_data:
            print(f"\nSuccessfully processed {len(processed_data)} files for measure {measure}")
            combined_data = pd.concat(processed_data)
            combined_data['Date'] = pd.to_datetime(combined_data['Date'])
            combined_data.set_index('Date', inplace=True)

            print("\nData shape:", combined_data.shape)
            print("\nUnique dates:", len(combined_data.index.unique()))
            print("Number of funds:", len(combined_data['Source_File'].unique()))
            print("Funds:", combined_data['Source_File'].unique())

            # Prepare features
            features = pd.DataFrame({
                'Value': combined_data[measure],
                'Rolling_Mean_12': combined_data[f'{measure}_Rolling_Mean_12'],
                'Rolling_Std_12': combined_data[f'{measure}_Rolling_Std_12'],
                'Rolling_Mean_24': combined_data[f'{measure}_Rolling_Mean_24'],
                'Rolling_Std_24': combined_data[f'{measure}_Rolling_Std_24'],
                'Fund': combined_data['Source_File'],
                'Anomaly_Label_12': combined_data[f'{measure}_Dynamic_Outlier_12'].astype(int),
                'Anomaly_Label_24': combined_data[f'{measure}_Dynamic_Outlier_24'].astype(int),
            })

            features = features.dropna()
            print(f"\nFeatures shape after preprocessing: {features.shape}")

            # Prepare data for modeling
            X = features[['Value', 'Rolling_Mean_12', 'Rolling_Std_12', 'Rolling_Mean_24', 'Rolling_Std_24']]
            y = features['Anomaly_Label_12']

            print("\nClass distribution:")
            print(y.value_counts(normalize=True))

            # Scale features
            scaler = StandardScaler()
            X_scaled = scaler.fit_transform(X)
            X_scaled = pd.DataFrame(X_scaled, columns=X.columns, index=X.index)

            # Define classifiers
            classifiers = {
                'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
                'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
                'SVM': SVC(probability=True, random_state=42),
                'Gradient Boosting': GradientBoostingClassifier(random_state=42)
            }

            for name, clf in classifiers.items():
                print(f"\nPerforming k-fold analysis for {name} on {measure}...")

                kfold_results = perform_kfold_analysis(X_scaled, y, clf)
                kfold_results.update({
                    'Model': name,
                    'Measure': measure,
                    'Dataset_Size': len(X),
                    'Positive_Class_Ratio': y.mean()
                })

                all_results.append(kfold_results)
                print(f"Results for {name}:")
                for key, value in kfold_results.items():
                    if isinstance(value, (int, float)):
                        print(f"{key}: {value:.4f}")
                    else:
                        print(f"{key}: {value}")

            print(f"\nCompleted analysis for measure: {measure}")
        else:
            print(f"No data processed for measure: {measure}. Please check the input files.")

    if all_results:
        print("\nSaving final results...")
        final_results_df = pd.DataFrame(all_results)
        final_results_path = os.path.join(folder_path, "Comprehensive_Model_Comparison.xlsx")

        with pd.ExcelWriter(final_results_path, engine='openpyxl') as writer:
            final_results_df.to_excel(writer, sheet_name='Overall_Results', index=False)

            pivot_measure = final_results_df.pivot_table(
                index='Model',
                columns='Measure',
                values=['test_roc_auc', 'test_accuracy', 'test_f1'],
                aggfunc='mean'
            )
            pivot_measure.to_excel(writer, sheet_name='Summary_by_Measure')

            pivot_model = final_results_df.pivot_table(
                index='Measure',
                columns='Model',
                values=['test_roc_auc', 'test_accuracy', 'test_f1'],
                aggfunc='mean'
            )
            pivot_model.to_excel(writer, sheet_name='Summary_by_Model')

        print(f"Results saved to {final_results_path}")
    else:
        print("No results were generated. Please check the input data and processing steps.")

if __name__ == "__main__":
    main()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted successfully
Checking folder contents...
Found file: Pictet.xlsx
Found file: Pictet.csv
Found file: creditsuisse.csv
Found file: creditsuisse.xlsx
Found file: 3645.csv
Found file: 3645.xlsx
Found file: Finreon.xlsx
Found file: Finreon.csv
Found file: SGKB.xlsx
Found file: SGKB.csv
Found file: IAM.xlsx
Found file: IAM.csv
Found file: 21216.xlsx
Found file: 21216.csv
Found file: Vontobel.csv
Found file: Vontobel.xlsx
Found file: zCapital.csv
Found file: zCapital.xlsx
Found file: GAM.csv
Found file: GAM.xlsx
Found file: Lo.xlsx
Found file: Lo.csv
Found file: SaraSelect.xlsx
Found file: SaraSelect.csv
Found file: analysis_output
Found file: analysis_plots
Found file: Model_Comparison_Alpha__annualisiert_.xlsx
Found file: Model_Comparison_Value_Growth.xlsx
Found file: Model_Comparison_Small_Large.xlsx
Found file: Model_Comparison_Momentum.xlsx

/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.p

Results for Logistic Regression:
test_accuracy: 0.9010
test_accuracy_std: 0.0154
test_precision: 0.0000
test_precision_std: 0.0000
test_recall: 0.0000
test_recall_std: 0.0000
test_f1: 0.0000
test_f1_std: 0.0000
test_roc_auc: 0.6030
test_roc_auc_std: 0.0715
train_accuracy: 0.9010
train_precision: 0.0000
train_recall: 0.0000
train_f1: 0.0000
train_roc_auc: 0.6288
Model: Logistic Regression
Measure: Alpha..annualisiert.
Dataset_Size: 1081.0000
Positive_Class_Ratio: 0.0990

Performing k-fold analysis for SVM on Alpha..annualisiert....


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.p

Results for SVM:
test_accuracy: 0.9010
test_accuracy_std: 0.0154
test_precision: 0.0000
test_precision_std: 0.0000
test_recall: 0.0000
test_recall_std: 0.0000
test_f1: 0.0000
test_f1_std: 0.0000
test_roc_auc: 0.9013
test_roc_auc_std: 0.0496
train_accuracy: 0.9019
train_precision: 0.6000
train_recall: 0.0092
train_f1: 0.0180
train_roc_auc: 0.9506
Model: SVM
Measure: Alpha..annualisiert.
Dataset_Size: 1081.0000
Positive_Class_Ratio: 0.0990

Performing k-fold analysis for Gradient Boosting on Alpha..annualisiert....
Results for Gradient Boosting:
test_accuracy: 0.9029
test_accuracy_std: 0.0162
test_precision: 0.6357
test_precision_std: 0.2216
test_recall: 0.1098
test_recall_std: 0.0306
test_f1: 0.1812
test_f1_std: 0.0431
test_roc_auc: 0.7287
test_roc_auc_std: 0.0413
train_accuracy: 0.9598
train_precision: 1.0000
train_recall: 0.5951
train_f1: 0.7446
train_roc_auc: 0.9961
Model: Gradient Boosting
Measure: Alpha..annualisiert.
Dataset_Size: 1081.0000
Positive_Class_Ratio: 0.0990

Completed 

...

Results for SVM:
test_accuracy: 0.9047
test_accuracy_std: 0.0086
test_precision: 0.2000
test_precision_std: 0.4000
test_recall: 0.0100
test_recall_std: 0.0200
test_f1: 0.0190
test_f1_std: 0.0381
test_roc_auc: 0.9178
test_roc_auc_std: 0.0299
train_accuracy: 0.9068
train_precision: 0.8000
train_recall: 0.0309
train_f1: 0.0588
train_roc_auc: 0.9530
Model: SVM
Measure: Momentum
Dataset_Size: 1081.0000
Positive_Class_Ratio: 0.0962

Performing k-fold analysis for Gradient Boosting on Momentum...
Results for Gradient Boosting:
test_accuracy: 0.9093
test_accuracy_std: 0.0104
test_precision: 0.7033
test_precision_std: 0.1694
test_recall: 0.1168
test_recall_std: 0.0285
test_f1: 0.1977
test_f1_std: 0.0409
test_roc_auc: 0.7382
test_roc_auc_std: 0.0579
train_accuracy: 0.9621
train_precision: 1.0000
train_recall: 0.6053
train_f1: 0.7534
train_roc_auc: 0.9983
Model: Gradient Boosting
Measure: Momentum
Dataset_Size: 1081.0000
Positive_Class_Ratio: 0.0962

Completed analysis for measure: Momentum

Proc

/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.p

Results for Logistic Regression:
test_accuracy: 0.9001
test_accuracy_std: 0.0134
test_precision: 0.0000
test_precision_std: 0.0000
test_recall: 0.0000
test_recall_std: 0.0000
test_f1: 0.0000
test_f1_std: 0.0000
test_roc_auc: 0.5804
test_roc_auc_std: 0.0831
train_accuracy: 0.9001
train_precision: 0.0000
train_recall: 0.0000
train_f1: 0.0000
train_roc_auc: 0.6145
Model: Logistic Regression
Measure: Volatility
Dataset_Size: 1081.0000
Positive_Class_Ratio: 0.0999

Performing k-fold analysis for SVM on Volatility...


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.p

Results for SVM:
test_accuracy: 0.9001
test_accuracy_std: 0.0134
test_precision: 0.0000
test_precision_std: 0.0000
test_recall: 0.0000
test_recall_std: 0.0000
test_f1: 0.0000
test_f1_std: 0.0000
test_roc_auc: 0.9100
test_roc_auc_std: 0.0249
train_accuracy: 0.9013
train_precision: 0.8000
train_recall: 0.0118
train_f1: 0.0232
train_roc_auc: 0.9368
Model: SVM
Measure: Volatility
Dataset_Size: 1081.0000
Positive_Class_Ratio: 0.0999

Performing k-fold analysis for Gradient Boosting on Volatility...
Results for Gradient Boosting:
test_accuracy: 0.9020
test_accuracy_std: 0.0221
test_precision: 0.4167
test_precision_std: 0.3333
test_recall: 0.1236
test_recall_std: 0.1212
test_f1: 0.1888
test_f1_std: 0.1781
test_roc_auc: 0.7615
test_roc_auc_std: 0.0716
train_accuracy: 0.9639
train_precision: 1.0000
train_recall: 0.6405
train_f1: 0.7796
train_roc_auc: 0.9978
Model: Gradient Boosting
Measure: Volatility
Dataset_Size: 1081.0000
Positive_Class_Ratio: 0.0999

Completed analysis for measure: Volatili

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline

def process_measures(df, measures, window_size):
    """Process multiple measures with specified window size."""
    for measure in measures:
        df[f'{measure}_Rolling_Mean_{window_size}'] = df[measure].rolling(window=window_size, min_periods=1).mean()
        df[f'{measure}_Rolling_Std_{window_size}'] = df[measure].rolling(window=window_size, min_periods=1).std()
        df[f'{measure}_Dynamic_Outlier_{window_size}'] = (
            (df[measure] - df[f'{measure}_Rolling_Mean_{window_size}']).abs() >
            2 * df[f'{measure}_Rolling_Std_{window_size}']
        )

    # Create combined anomaly indicator
    outlier_cols = [f'{measure}_Dynamic_Outlier_{window_size}' for measure in measures]
    df[f'Combined_Outlier_{window_size}'] = df[outlier_cols].any(axis=1)

    return df

def create_feature_matrices(df, measures, window_size):
    """Create feature matrix and combined target."""
    feature_cols = []
    for measure in measures:
        feature_cols.extend([
            measure,
            f'{measure}_Rolling_Mean_{window_size}',
            f'{measure}_Rolling_Std_{window_size}'
        ])
    X = df[feature_cols].copy()
    y = df[f'Combined_Outlier_{window_size}'].astype(int)

    print("\nFeature distribution:")
    print(X.describe())
    print("\nTarget distribution:")
    print(y.value_counts(normalize=True))

    return X, y

def analyze_single_file(file_path, measures, window_sizes=[12, 24]):
    """Analyze a single file."""
    try:
        # Read and process file
        df = pd.read_excel(file_path, sheet_name="FactorOutput")
        filename = os.path.basename(file_path)
        print(f"\nProcessing file: {filename}")
        print(f"Data shape: {df.shape}")

        if 'Date' in df.columns:
            df['Date'] = pd.to_datetime(df['Date'])
            print(f"Date range: {df['Date'].min()} to {df['Date'].max()}")
        else:
            raise ValueError("The 'Date' column is missing in the file.")

        df.set_index('Date', inplace=True)
        results_all = []

        # Process for each window size
        for window_size in window_sizes:
            print(f"\nWindow size: {window_size} months")

            # Process measures
            df_processed = process_measures(df.copy(), measures, window_size)

            # Create feature matrices
            X, y = create_feature_matrices(df_processed, measures, window_size)

            # Print outlier counts per factor
            print("\nOutliers per factor:")
            for measure in measures:
                outlier_count = df_processed[f'{measure}_Dynamic_Outlier_{window_size}'].sum()
                print(f"{measure}: {outlier_count} outliers ({outlier_count/len(df_processed)*100:.1f}%)")

            # Split data (used for final predictions, not k-fold)
            X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

            # Initialize classifiers
            classifiers = {
                'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
                'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
                'SVM': SVC(probability=True, random_state=42),
                'Gradient Boosting': GradientBoostingClassifier(random_state=42)
            }

            # Train and evaluate each classifier
            for name, clf in classifiers.items():
                print(f"\nTraining {name} with k-fold cross-validation...")

                # Define pipeline
                pipeline = Pipeline([
                    ('imputer', SimpleImputer(strategy='mean')),
                    ('scaler', StandardScaler()),
                    ('smote', SMOTE(random_state=42, k_neighbors=1)),
                    ('classifier', clf)
                ])

                # Execute k-fold cross-validation
                print(f"Running k-fold cross-validation for {name}...")
                cv_scores = cross_val_score(pipeline, X, y, cv=5, scoring='roc_auc')

                # Fit the model on the full training data
                pipeline.fit(X_train, y_train)

                # Evaluate on test data
                y_pred = pipeline.predict(X_test)
                y_proba = pipeline.predict_proba(X_test)[:, 1]
                report = classification_report(y_test, y_pred, zero_division=0, output_dict=True)
                roc_auc = roc_auc_score(y_test, y_proba)

                # Append results
                results_all.append({
                    'File': filename,
                    'Model': name,
                    'Window': window_size,
                    'Accuracy': report['accuracy'],
                    'Precision': report['1']['precision'],
                    'Recall': report['1']['recall'],
                    'F1-Score': report['1']['f1-score'],
                    'ROC-AUC': roc_auc,
                    'CV Mean ROC-AUC': cv_scores.mean(),
                    'CV Std ROC-AUC': cv_scores.std()
                })

                # Print cross-validation results
                print(f"{name} Mean ROC-AUC: {cv_scores.mean():.3f} (+/- {cv_scores.std() * 2:.3f})")

        return pd.DataFrame(results_all)

    except Exception as e:
        print(f"Error processing file {file_path}: {e}")
        return None

# List of factors to process
factors = ['Alpha..annualisiert.', 'Value.Growth', 'Small.Large', 'Momentum', 'Volatility']

# Process each file individually
all_results = []
for filename in os.listdir(folder_path):
    if filename.endswith(".xlsx") and not filename.startswith("Model_Comparison"):
        file_path = os.path.join(folder_path, filename)
        results = analyze_single_file(file_path, factors)
        if results is not None:
            all_results.append(results)

if all_results:
    # Combine and display results
    final_results = pd.concat(all_results, ignore_index=True)
    print("\nFinal Results:")
    print("\nAverage performance by model and window size:")
    avg_results = final_results.groupby(['Model', 'Window']).agg({
        'Accuracy': ['mean', 'std'],
        'Precision': ['mean', 'std'],
        'Recall': ['mean', 'std'],
        'F1-Score': ['mean', 'std'],
        'ROC-AUC': ['mean', 'std']
    }).round(3)
    print(avg_results)

    print("\nPerformance by file:")
    file_results = final_results.groupby('File').agg({
        'Accuracy': 'mean',
        'Precision': 'mean',
        'Recall': 'mean',
        'F1-Score': 'mean',
        'ROC-AUC': 'mean'
    }).round(3)
    print(file_results)

    # Save results
    results_path = os.path.join(folder_path, "Individual_Model_Comparison.xlsx")
    final_results.to_excel(results_path, index=False)
    print(f"\nDetailed results saved to {results_path}")
else:
    print("No files processed successfully.")



Processing file: Pictet.xlsx
Data shape: (91, 6)
Date range: 2016-09-30 12:00:00 to 2024-03-31 12:00:00

Window size: 12 months

Feature distribution:
       Alpha..annualisiert.  Alpha..annualisiert._Rolling_Mean_12  \
count             91.000000                             91.000000   
mean               0.000485                              0.003359   
std                0.027605                              0.023751   
min               -0.045673                             -0.037546   
25%               -0.025118                             -0.011173   
50%                0.000249                              0.007535   
75%                0.025836                              0.022476   
max                0.050992                              0.038199   

       Alpha..annualisiert._Rolling_Std_12  Value.Growth  \
count                            90.000000     91.000000   
mean                              0.010496     -0.113990   
std                               0.006644    

/usr/local/lib/python3.10/dist-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/utils/_tags.py:354: FutureWarning: The SMOTE or classes from which it inherits use `_get_tags` and `_more_tags`. Please define the `__sklearn_tags__` method, or inherit from `sklearn.base.BaseEstimator` and/or other appropriate mixins such as `sklearn.base.TransformerMixin`, `sklearn.base.ClassifierMixin`, `sklearn.base.RegressorMixin`, and `sklearn.base.OutlierMixin`. From scikit-learn 1.7, not defining `__sklearn_tags__` will raise an error.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validatio

Random Forest Mean ROC-AUC: 0.314 (+/- 0.172)

Training Logistic Regression with k-fold cross-validation...
Running k-fold cross-validation for Logistic Regression...
Logistic Regression Mean ROC-AUC: 0.357 (+/- 0.190)

Training SVM with k-fold cross-validation...
Running k-fold cross-validation for SVM...


/usr/local/lib/python3.10/dist-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/utils/_tags.py:354: FutureWarning: The SMOTE or classes from which it inherits use `_get_tags` and `_more_tags`. Please define the `__sklearn_tags__` method, or inherit from `sklearn.base.BaseEstimator` and/or other appropriate mixins such as `sklearn.base.TransformerMixin`, `sklearn.base.ClassifierMixin`, `sklearn.base.RegressorMixin`, and `sklearn.base.OutlierMixin`. From scikit-learn 1.7, not defining `__sklearn_tags__` will raise an error.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validatio

SVM Mean ROC-AUC: 0.348 (+/- 0.250)

Training Gradient Boosting with k-fold cross-validation...
Running k-fold cross-validation for Gradient Boosting...


...

Random Forest Mean ROC-AUC: 0.436 (+/- 0.506)

Training Logistic Regression with k-fold cross-validation...
Running k-fold cross-validation for Logistic Regression...


/usr/local/lib/python3.10/dist-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/utils/_tags.py:354: FutureWarning: The SMOTE or classes from which it inherits use `_get_tags` and `_more_tags`. Please define the `__sklearn_tags__` method, or inherit from `sklearn.base.BaseEstimator` and/or other appropriate mixins such as `sklearn.base.TransformerMixin`, `sklearn.base.ClassifierMixin`, `sklearn.base.RegressorMixin`, and `sklearn.base.OutlierMixin`. From scikit-learn 1.7, not defining `__sklearn_tags__` will raise an error.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validatio

Logistic Regression Mean ROC-AUC: 0.239 (+/- 0.285)

Training SVM with k-fold cross-validation...
Running k-fold cross-validation for SVM...
SVM Mean ROC-AUC: 0.565 (+/- 0.548)

Training Gradient Boosting with k-fold cross-validation...
Running k-fold cross-validation for Gradient Boosting...


/usr/local/lib/python3.10/dist-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/utils/_tags.py:354: FutureWarning: The SMOTE or classes from which it inherits use `_get_tags` and `_more_tags`. Please define the `__sklearn_tags__` method, or inherit from `sklearn.base.BaseEstimator` and/or other appropriate mixins such as `sklearn.base.TransformerMixin`, `sklearn.base.ClassifierMixin`, `sklearn.base.RegressorMixin`, and `sklearn.base.OutlierMixin`. From scikit-learn 1.7, not defining `__sklearn_tags__` will raise an error.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validatio

Gradient Boosting Mean ROC-AUC: 0.461 (+/- 0.275)
Error processing file /content/drive/My Drive/Factordata/Individual_Model_Comparison.xlsx: Worksheet named 'FactorOutput' not found
Error processing file /content/drive/My Drive/Factordata/Comprehensive_Model_Comparison.xlsx: Worksheet named 'FactorOutput' not found
Error processing file /content/drive/My Drive/Factordata/Cross_Validation_Results.xlsx: Worksheet named 'FactorOutput' not found

Final Results:

Average performance by model and window size:
                           Accuracy        Precision        Recall         \
                               mean    std      mean    std   mean    std   
Model               Window                                                  
Gradient Boosting   12        0.674  0.057     0.363  0.155  0.343  0.186   
                    24        0.758  0.095     0.672  0.162  0.695  0.131   
Logistic Regression 12        0.618  0.056     0.352  0.085  0.459  0.194   
                    24       

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, cross_val_score, KFold, cross_validate
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline

def perform_kfold_analysis(X, y, pipeline, n_splits=5):
    """
    Perform comprehensive k-fold cross validation with multiple metrics.
    """
    scoring = {
        'accuracy': 'accuracy',
        'precision': 'precision',
        'recall': 'recall',
        'f1': 'f1',
        'roc_auc': 'roc_auc'
    }

    kfold = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    cv_results = cross_validate(
        pipeline, X, y,
        cv=kfold,
        scoring=scoring,
        return_train_score=True,
        n_jobs=-1  # Use all available cores
    )

    # Calculate mean and std for each metric
    results = {}
    for metric in scoring.keys():
        # Test metrics
        results[f'test_{metric}'] = cv_results[f'test_{metric}'].mean()
        results[f'test_{metric}_std'] = cv_results[f'test_{metric}'].std()
        # Train metrics
        results[f'train_{metric}'] = cv_results[f'train_{metric}'].mean()
        results[f'train_{metric}_std'] = cv_results[f'train_{metric}'].std()

    return results

def process_measures(df, measures, window_size):
    """Process multiple measures with specified window size."""
    for measure in measures:
        df[f'{measure}_Rolling_Mean_{window_size}'] = df[measure].rolling(window=window_size, min_periods=1).mean()
        df[f'{measure}_Rolling_Std_{window_size}'] = df[measure].rolling(window=window_size, min_periods=1).std()
        df[f'{measure}_Dynamic_Outlier_{window_size}'] = (
            (df[measure] - df[f'{measure}_Rolling_Mean_{window_size}']).abs() >
            2 * df[f'{measure}_Rolling_Std_{window_size}']
        )

    outlier_cols = [f'{measure}_Dynamic_Outlier_{window_size}' for measure in measures]
    df[f'Combined_Outlier_{window_size}'] = df[outlier_cols].any(axis=1)
    return df

def analyze_single_file(file_path, measures, window_sizes=[12, 24]):
    """Analyze a single file with enhanced cross-validation."""
    try:
        df = pd.read_excel(file_path, sheet_name="FactorOutput")
        filename = os.path.basename(file_path)
        print(f"\nProcessing file: {filename}")
        print(f"Data shape: {df.shape}")

        if 'Date' in df.columns:
            df['Date'] = pd.to_datetime(df['Date'])
            print(f"Date range: {df['Date'].min()} to {df['Date'].max()}")
        else:
            raise ValueError("The 'Date' column is missing in the file.")

        df.set_index('Date', inplace=True)
        results_all = []

        for window_size in window_sizes:
            print(f"\nAnalyzing window size: {window_size} months")
            df_processed = process_measures(df.copy(), measures, window_size)

            # Create feature matrices
            feature_cols = []
            for measure in measures:
                feature_cols.extend([
                    measure,
                    f'{measure}_Rolling_Mean_{window_size}',
                    f'{measure}_Rolling_Std_{window_size}'
                ])
            X = df_processed[feature_cols].copy()
            y = df_processed[f'Combined_Outlier_{window_size}'].astype(int)

            # Print class distribution
            print("\nClass distribution:")
            print(y.value_counts(normalize=True))

            # Define classifiers
            classifiers = {
                'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
                'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
                'SVM': SVC(probability=True, random_state=42),
                'Gradient Boosting': GradientBoostingClassifier(random_state=42)
            }

            for name, clf in classifiers.items():
                print(f"\nPerforming cross-validation analysis for {name}...")

                # Create pipeline
                pipeline = Pipeline([
                    ('imputer', SimpleImputer(strategy='mean')),
                    ('scaler', StandardScaler()),
                    ('smote', SMOTE(random_state=42, k_neighbors=1)),
                    ('classifier', clf)
                ])

                # Perform k-fold analysis
                cv_results = perform_kfold_analysis(X, y, pipeline)

                # Add metadata
                cv_results.update({
                    'File': filename,
                    'Model': name,
                    'Window': window_size,
                    'Dataset_Size': len(X),
                    'Positive_Class_Ratio': y.mean()
                })

                # Print detailed results
                print(f"\nResults for {name}:")
                for key, value in cv_results.items():
                    if isinstance(value, (int, float)):
                        print(f"{key}: {value:.4f}")
                    else:
                        print(f"{key}: {value}")

                results_all.append(cv_results)

        return pd.DataFrame(results_all)

    except Exception as e:
        print(f"Error processing file {file_path}: {e}")
        return None

# Main execution
factors = ['Alpha..annualisiert.', 'Value.Growth', 'Small.Large', 'Momentum', 'Volatility']

# Process each file
all_results = []
for filename in os.listdir(folder_path):
    if filename.endswith(".xlsx") and not filename.startswith("Model_Comparison"):
        file_path = os.path.join(folder_path, filename)
        results = analyze_single_file(file_path, factors)
        if results is not None:
            all_results.append(results)

if all_results:
    # Combine results
    final_results = pd.concat(all_results, ignore_index=True)

    # Create summary statistics
    print("\nSummary Statistics:")

    # By model and window size
    model_window_stats = final_results.groupby(['Model', 'Window']).agg({
        'test_accuracy': ['mean', 'std'],
        'test_precision': ['mean', 'std'],
        'test_recall': ['mean', 'std'],
        'test_f1': ['mean', 'std'],
        'test_roc_auc': ['mean', 'std']
    }).round(4)

    # By file
    file_stats = final_results.groupby('File').agg({
        'test_accuracy': 'mean',
        'test_precision': 'mean',
        'test_recall': 'mean',
        'test_f1': 'mean',
        'test_roc_auc': 'mean'
    }).round(4)

    # Save detailed results
    result_path = os.path.join(folder_path, "Cross_Validation_Results.xlsx")
    with pd.ExcelWriter(result_path, engine='openpyxl') as writer:
        # Save overall results
        final_results.to_excel(writer, sheet_name='Detailed_Results', index=False)

        # Save summary statistics
        model_window_stats.to_excel(writer, sheet_name='Model_Window_Summary')
        file_stats.to_excel(writer, sheet_name='File_Summary')

        # Create performance comparison across models
        pivot_model = final_results.pivot_table(
            index='File',
            columns=['Model', 'Window'],
            values=['test_roc_auc', 'test_accuracy', 'test_f1'],
            aggfunc='mean'
        ).round(4)
        pivot_model.to_excel(writer, sheet_name='Performance_Comparison')

    print(f"\nResults saved to {result_path}")

    # Print summary statistics
    print("\nModel & Window Size Performance:")
    print(model_window_stats)
    print("\nFile Performance:")
    print(file_stats)
else:
    print("No files processed successfully.")


Processing file: Pictet.xlsx
Data shape: (91, 6)
Date range: 2016-09-30 12:00:00 to 2024-03-31 12:00:00

Analyzing window size: 12 months

Class distribution:
Combined_Outlier_12
0    0.78022
1    0.21978
Name: proportion, dtype: float64

Performing cross-validation analysis for Random Forest...

Results for Random Forest:
test_accuracy: 0.7573
test_accuracy_std: 0.0848
train_accuracy: 1.0000
train_accuracy_std: 0.0000
test_precision: 0.3700
test_precision_std: 0.2182
train_precision: 1.0000
train_precision_std: 0.0000
test_recall: 0.4586
test_recall_std: 0.3719
train_recall: 1.0000
train_recall_std: 0.0000
test_f1: 0.3467
test_f1_std: 0.2238
train_f1: 1.0000
train_f1_std: 0.0000
test_roc_auc: 0.7576
test_roc_auc_std: 0.1729
train_roc_auc: 1.0000
train_roc_auc_std: 0.0000
File: Pictet.xlsx
Model: Random Forest
Window: 12.0000
Dataset_Size: 91.0000
Positive_Class_Ratio: 0.2198

Performing cross-validation analysis for Logistic Regression...

Results for Logistic Regression:
test_accura